# status

> Feedback and loading states: alerts, toasts, progress bars, spinners and skeletons

In [ ]:
#| default_exp status

An interface needs to explain what happened and what is still happening. These components show that feedback but don't run the operation. The application owns the state; the markup describes it.

In [ ]:
from fasthtml.common import Div, Span, H3, P, Label, Template, unqid
from fastcore.xml import FT
from fasthtml.core import FtResponse
from answerui.registry import reg, hypelt
from answerui.controls import Button
from answerui.surfaces import Card
from answerui.layout import Stack
from answerui.demo import example
import answerui.base  # the reset and tokens this styling is written on

In [ ]:
from fasthtml.common import Form
from answerui.demo import demo
from fastcore.test import *

## Notice

An alert and a toast are both a notice: a stack of content beside an optional dismiss button. The stack fills the remaining width. The dismiss button keeps its natural width and has the tighter padding of the `compact` button variant. `Alert` and `Toast` each add their own background and border to the row that the `notice` rule lays out:


In [ ]:
reg.rule(r"""
:is([alert], [toast]) { display: flex; align-items: flex-start; gap: var(--space);
  > [stack] { flex: 1; min-inline-size: 0; }
  > button[dismiss] { flex-shrink: 0; --pad-block: var(--space-3); --pad-inline: var(--space-2); } }
""", layer='components', doc='The row shared by alerts and toasts: the message beside its dismiss button.')

def _notice(c, dismissible, **attrs):
    "A stack of content beside an optional dismiss button"
    close = Button('Dismiss', type='button', secondary=True, dismiss=True) if dismissible else None
    return Div(Stack(*c, gap='var(--space-2)'), close, **attrs)

## Alert

An alert groups a message and any supporting content. It is drawn as a card, coloured like a badge: a soft shade of the accent hue, or of the hue that a severity attribute such as `warning` selects. With `dismissible=True`, the alert has a dismiss button that removes the whole alert, not just its text. Because the click handler is delegated, it also works on alerts that htmx inserts later. The handler is the alert's `dismiss` rule. To hide the alert instead of removing it, replace the rule with `Alert.rule('dismiss', ...)`.

In [ ]:
_alert_css = r"""
background: var(--v-3); color: var(--v2); border-color: var(--v-2);
"""

@hypelt(sel='[alert]', layer='components', css=_alert_css)
def Alert(
    *c,                # The message and any supporting content
    dismissible=False, # Include a button that removes this alert
    **attrs,           # Attributes for the containing div, including severity and role
):
    "An in-page message with optional dismissal"
    return _notice(c, dismissible, card=True, alert=True, **attrs)

Alert.rule('dismiss', on={('& > button[dismiss]', 'click'): "el.closest('[alert]').remove()"}, layer='components', doc='The dismiss button removes the alert.')

Alert.dismiss [components] [alert]: The dismiss button removes the alert.

An alert's colour and whether a screen reader announces it are separate decisions. `Alert` adds no `role` by default, because a persistent warning need not interrupt a screen-reader user. For polite feedback, establish an empty `role="status"` region in the page, then insert or update the message inside it. Reserve `role="alert"` for urgent messages. A screen reader may not announce a message when a single swap creates the status region with the message already inside it; see [ARIA live regions](https://developer.mozilla.org/en-US/docs/Web/Accessibility/ARIA/Guides/Live_regions).

In [ ]:
@example(Alert)
def _alert_example():
    return Stack(
        Alert('Your changes have been saved.', success=True, dismissible=True),
        Alert(H3('Storage is nearly full'), P('Remove unused files or choose a larger plan.'), warning=True),
        Alert('The upload failed. Your original file has not changed.', danger=True, dismissible=True))

In [ ]:
Alert.demo()

```html
<div stack>
  <div card alert success>
    <div stack style="--gap: var(--space-2);">Your changes have been saved.</div>
<button type="button" secondary dismiss>Dismiss</button>  </div>
  <div card alert warning>
    <div stack style="--gap: var(--space-2);">
      <h3>Storage is nearly full</h3>
      <p>Remove unused files or choose a larger plan.</p>
    </div>
  </div>
  <div card alert danger>
    <div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div>
<button type="button" secondary dismiss>Dismiss</button>  </div>
</div>

```

Alerts have no timer: important information should not disappear before someone has read it.

## Toast

Place an empty `Toasts()` region in the page before delivering messages to it. It is a polite live region and a manual popover. Its `init` rule opens the popover when the region arrives in the page. The browser draws an open popover above ordinary page content without any `z-index`, and a manual popover stays open when the user clicks outside it or presses Escape. A modal dialog opened later appears above the region. Feedback needed while a modal is active belongs inside that dialog.

By default, a toast hides after five seconds. Its lifetime is a CSS animation lasting `--life`, whose last frame sets `display: none`. `Toast` sets `--life` from its `life` argument. The animation pauses while the pointer is over the toast or keyboard focus is inside it, and resumes where it stopped. No script keeps time. A hidden toast stays in the region's markup. With `life=None`, `Toast` sets no `--life`. Without it the `animation` declaration is invalid, and the toast stays until dismissed. In a toast, `--motion` sets only the duration of the entry transition. When the user prefers reduced motion, `--motion` is `0ms`, and the toast appears with no transition but keeps its full lifetime. The dismiss button's handler is the toast's `dismiss` rule. To change how a toast is dismissed, replace the rule with `Toast.rule('dismiss', ...)`.

In [ ]:
_toasts_on = {('&', 'init'): "if (!el.matches(':popover-open')) el.showPopover()"}
_toasts_css = r"""
&[popover] {
  position: fixed; inset: auto; inset-block-start: var(--space); inset-inline-end: var(--space); margin: 0;
  padding: var(--space-3); border: 0; background: none; box-shadow: none; overflow: visible; color: inherit; pointer-events: none;
  inline-size: min(22rem, calc(100vw - 2 * var(--space)));
  &:popover-open { display: grid; gap: var(--space-2); }
}
> [toast] { pointer-events: auto; }
"""

@hypelt(sel='[toasts]', layer='components', on=_toasts_on, css=_toasts_css)
def Toasts(*c, **attrs):
    "The page's polite live region for toast deliveries"
    return Div(*c, popover='manual', role='status', aria_live='polite', aria_atomic='false',
        aria_relevant='additions text', toasts=True, **attrs)

In [ ]:
_toast_css = r"""
--face: var(--n-2); background: var(--face); color: var(--n2); border: 1px solid var(--n-1);
border-radius: var(--radius); padding: var(--space-2) var(--space); box-shadow: var(--floating);
transition: opacity var(--motion) var(--ease), translate var(--motion) var(--ease);
@starting-style { opacity: 0; translate: 0 calc(-1 * var(--space-2)); }
animation: aui-toast-life var(--life) linear forwards;
&:is(:hover, :focus-within) { animation-play-state: paused; }
&:is([success], [warning], [danger], [info]) { --face: var(--v-3); color: var(--v2); border-color: var(--v-2); }
"""

@hypelt(sel='[toast]', layer='components', css=_toast_css)
def Toast(
    *c,               # The message
    life=5000,        # Unattended milliseconds before it hides; None to stay until dismissed
    dismissible=True, # Include a dismiss button
    oob=False,        # Wrap for out-of-band delivery into the page's region, beside a fragment
    **attrs,          # Attributes for the message, including severity
):
    "A brief message for the toast region"
    if life: attrs['style'] = f'--life: {life}ms; ' + attrs.get('style', '')
    toast = _notice(c, dismissible, toast=True, **attrs)
    return Div(toast, hx_swap_oob='beforeend:[toasts]') if oob else toast

Toast.rule('dismiss', on={('& > button[dismiss]', 'click'): "el.closest('[toast]').remove()"}, layer='components', doc='The dismiss button removes the toast at once.')

reg.rule(r'''
/* The lifetime is a timer that ends by hiding the toast. Without --life the shorthand is invalid, and the toast has no animation. */
@keyframes aui-toast-life { to { display: none; } }
''', layer='components')

Toast.dismiss [components] [toast]: The dismiss button removes the toast at once.

Deliver a toast by putting it **inside** an out-of-band wrapper: `Div(Toast('Saved'), hx_swap_oob='beforeend:[toasts]')`. `Toast('Saved', oob=True)` builds this wrapper. For a `beforeend` swap, htmx discards the wrapper and appends what is inside it: the complete toast, with its attributes and dismiss button. With `hx_swap_oob` on the toast itself, htmx would instead discard the toast's own element and append only its contents. See [out-of-band swaps](https://htmx.org/attributes/hx-swap-oob/).

In this example, the response is in a `template`, and the button's `hx-on:click` handler passes its HTML to [htmx.swap](https://htmx.org/api/#swap). This shows the delivery without a server, and without adding a behavior to the registry. In an app, a route returns the same wrapper, and the app needs no script of its own. Click to deliver both messages, then try hovering or focusing the timed one.

In [ ]:
@example(Toast)
def _toast_example():
    id = unqid()
    response = Div(Toast('Your changes have been saved.', success=True),
        Toast('The upload failed. Your original file has not changed.', life=None, danger=True),
        hx_swap_oob='beforeend:[toasts]')
    return Div(Button('Deliver notifications', type='button', secondary=True,
        hx_on_click=f"htmx.swap(this, document.getElementById('{id}').innerHTML, {{swapStyle: 'none'}})"),
        Template(response, id=id), Toasts())

In [ ]:
Toast.demo(height='14rem')

```html
<div>
<button type="button" secondary hx-on-click="htmx.swap(this, document.getElementById('_T_RqJ7jKQxWkAg76udX-sQ').innerHTML, {swapStyle: 'none'})">Deliver notifications</button><template id="_T_RqJ7jKQxWkAg76udX-sQ">    <div hx-swap-oob="beforeend:[toasts]">
      <div toast success style="--life: 5000ms; ">
        <div stack style="--gap: var(--space-2);">Your changes have been saved.</div>
<button type="button" secondary dismiss>Dismiss</button>      </div>
      <div toast danger>
        <div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div>
<button type="button" secondary dismiss>Dismiss</button>      </div>
    </div>
</template>  <div popover="manual" role="status" aria-live="polite" aria-atomic="false" aria-relevant="additions text" toasts></div>
</div>

```

### Queued toasts

A route that redirects returns no markup that could carry a toast. A route that answers an htmx request returns a fragment, and its toast has to arrive beside that fragment. In both cases, queue the message in the session with `add_toast`, passing its severity as an attribute such as `success=True`. The app delivers the queue with the next page or fragment it renders. A full page renders the toasts in its region with `Toasts(*pop_toasts(sess))`. A fragment response includes `toasts_oob(sess)` beside its content, and htmx swaps the toasts into the region already on the page. To deliver queued toasts with every markup response without changing each route, call `setup_toasts(app)`. It installs an `after` hook that adds `toasts_oob(sess)` to the response. When a full page has already rendered `Toasts(*pop_toasts(sess))`, the queue is empty and the hook adds nothing.

In [ ]:
toasts_key = 'aui_toasts'

def add_toast(sess, msg, life=5000, **attrs):
    "Queue `msg` in session `sess` for the next page or fragment, with its severity in `attrs`"
    sess.setdefault(toasts_key, []).append((msg, life, attrs))

def pop_toasts(sess):
    "The toasts queued in session `sess`, as `Toast`s, clearing the queue"
    return [Toast(m, life=l, **a) for m,l,a in sess.pop(toasts_key, [])]

def toasts_oob(sess):
    "The toasts queued in session `sess`, in an out-of-band wrapper for delivery beside a fragment, clearing the queue; None when nothing is queued"
    toasts = pop_toasts(sess)
    return Div(*toasts, hx_swap_oob='beforeend:[toasts]') if toasts else None

The hook, `toast_after`, adds the queued toasts when a route returns markup or nothing. A route that only queues a toast returns nothing. The toasts stay queued for the next page after a redirect or a file response, because neither has markup to carry them:

In [ ]:
def toast_after(resp, req, sess):
    "FastHTML `after` hook that adds the queued toasts out of band to a markup or empty response"
    if sess.get(toasts_key) and (not resp or isinstance(resp, (tuple, FT, FtResponse))): req.injects.append(toasts_oob(sess))

def setup_toasts(app):
    "Deliver queued toasts out of band with every markup or empty response of `app`"
    app.after.append(toast_after)

Here we queue two messages as a route would, then take them with `toasts_oob`, in the wrapper that a fragment response carries. Taking them clears the queue in `sess`:

In [ ]:
sess = {}
add_toast(sess, 'Your changes have been saved.', success=True)
add_toast(sess, 'The upload failed. Your original file has not changed.', life=None, danger=True)
oob = toasts_oob(sess)
test_eq(sess, {})
test_eq([t.attrs.get('danger') for t in oob.children], [None, True])
oob

<div hx-swap-oob="beforeend:[toasts]"><div toast success style="--life: 5000ms; "><div stack style="--gap: var(--space-2);">Your changes have been saved.</div><button type="button" secondary dismiss>Dismiss</button></div><div toast danger><div stack style="--gap: var(--space-2);">The upload failed. Your original file has not changed.</div><button type="button" secondary dismiss>Dismiss</button></div></div>

## Progress

When the amount of work is known, a native `progress` element represents the fraction completed. Omitting `value` makes it indeterminate. We keep the browser’s rendering and animation, setting only its dimensions and accent colour. Wrap the bar in a `label` whose text names the operation.

In [ ]:
Progress = hypelt('Progress', layer='components', doc='Native progress; omit value when the amount completed is unknown.', css=r"""
inline-size: 100%; block-size: .75rem; vertical-align: middle; accent-color: var(--v);
""")

A labelled bar for a known amount of work, and an indeterminate one for an unknown amount:


In [ ]:
@example(Progress)
def _progress_example():
    return Stack(Label('Importing files', Progress(value=40, max=100), stack=True, style='--gap: var(--space-3)'),
        Label('Waiting for the server', Progress(), stack=True, style='--gap: var(--space-3)'))


In [ ]:
Progress.demo()

```html
<div stack>
<label stack style="--gap: var(--space-3)">Importing files<progress value="40" max="100"></progress></label><label stack style="--gap: var(--space-3)">Waiting for the server<progress></progress></label></div>

```

## Spinner

For a short wait, a spinner beside a few words is often enough. It uses the same `aui-spin` animation as busy buttons, and one turn takes five times `--motion`. With reduced motion, the ring stays still, and the label alone explains the wait. As with alerts, add a live-region role only where an announcement is wanted.

In [ ]:
@hypelt(sel='[spinner]', layer='components', css=r"""
display: inline-flex; align-items: center; gap: var(--space-2);
&::before {
  content: ''; flex-shrink: 0; inline-size: 1em; block-size: 1em;
  border: 2px solid currentColor; border-inline-end-color: transparent; border-radius: var(--radius-full);
  animation: aui-spin calc(var(--motion) * 5) linear infinite;
}
""")
def Spinner(label='Loading…', **attrs):
    "A small spinning ring with visible text explaining the wait"
    return Span(label, spinner=True, **attrs)

In [ ]:
@example(Spinner)
def _spinner_example():
    return Spinner('Checking the connection…')

In [ ]:
Spinner.demo()

```html
<span spinner>Checking the connection…</span>
```

## Skeleton

A skeleton reserves space for content that has not arrived. It is decorative, not a loading message: `Skeleton` sets `aria-hidden` to keep it out of the accessibility tree. By default it is as wide as its container and `1em` tall, like a line of text. Set other dimensions with ordinary CSS, such as a `style` attribute. We leave it still rather than adding another animation.

In [ ]:
@hypelt(sel='[skeleton]', layer='components', css=r"""
display: block; inline-size: 100%; block-size: 1em;
background: var(--n-1); border-radius: var(--radius-1);
""")
def Skeleton(**attrs):
    "A decorative placeholder for content that has not loaded"
    return Span(skeleton=True, aria_hidden='true', **attrs)

In [ ]:
@example(Skeleton)
def _skeleton_example():
    return Stack(P('Loading the project summary…'),
        Card(Skeleton(style='inline-size: 40%; block-size: 1.5em'), Skeleton(), Skeleton(style='inline-size: 75%'),
            aria_busy='true'))

In [ ]:
Skeleton.demo()

```html
<div stack>
  <p>Loading the project summary…</p>
  <div stack card aria-busy="true">
<span skeleton aria-hidden="true" style="inline-size: 40%; block-size: 1.5em"></span><span skeleton aria-hidden="true"></span><span skeleton aria-hidden="true" style="inline-size: 75%"></span>  </div>
</div>

```

The loading message is outside the busy region. When the content arrives, replace the placeholders and clear `aria-busy`, or replace the whole card. `Skeleton` does not make these changes itself.

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()